# Cleaning Data

- the objective of this notebook is to clean all the data collected
- firstly we clean each CSV file by:
    - removing all missing values
    - creating a JSON parser
    - changing all the models into ml ready data
- we will then create the final HexGate Archive


In [ ]:
import pandas as pd
import os
import scipy as sp
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
PILOT_CSV = "./ToBeCleaned.csv"
df = pd.read_csv(PILOT_CSV)

df.head(10)

In [ ]:
empty_cells = df[df.isna().any(axis=1)]

empty_cells.head(5)

In [ ]:
clean_cells = df.dropna()

clean_cells.head(10)

In [ ]:
import json
import re
import os
import lol_context as lc
import pandas as pd

df = lc.Champion

def get_cc_stats(champion_list):
    data_dragon = []

    for champion in champion_list:
        result = []
        with open("./en_US/champion/"+champion, mode="r", encoding="utf-8") as read_file:
            data_phoenix = json.load(read_file)

            # Collect All the Crowd Control stats
            total_cc_score = 0
            crowd_control = re.findall(r"<status>(.*?)</status>", str(data_phoenix))

            for cc in crowd_control:
                if "Stun" in cc or "Root" in cc:
                    total_cc_score +=3
                elif "Knock" in cc:
                    total_cc_score +=2
                else:
                    total_cc_score +=1

            result.append(total_cc_score)

            # Collect the champion key
            champ_id = data_phoenix['data'][champion.replace(".json", "")]['key']
            result.append(champ_id)

            # Collect the champion name
            champ_name = champion.replace(".json", "").lower()
            result.append(champ_name)

            # Collect champion stats
            # They are [attack, defense, magic, difficulty]
            champ_stats = list(data_phoenix['data'][champion.replace(".json", "")]['info'].values())
            result.append(champ_stats)

            # Collect champion tags
            tag_list = ['Fighter', 'Marksman', 'Tank', 'Assassin', 'Support', 'Mage']

            sub_list = data_phoenix['data'][champion.replace(".json", "")]['tags']
            result.append([1 if tag in sub_list else 0 for tag in tag_list])

        data_dragon.append(result)

    return data_dragon

x = get_cc_stats(os.listdir("./en_US/champion/"))


column_names = ["CC_SCORE", "CHAMP_ID", "CHAMPION_NAME", "CHAMPION_INFO", "CHAMPION_TAGS"]
df = pd.DataFrame(x, columns=column_names)

df.head(5)

In [ ]:
blue_team = clean_cells[[
    'BLUE_TEAM_TOP',
    'BLUE_TEAM_JNG',
    'BLUE_TEAM_MID',
    'BLUE_TEAM_ADC',
    'BLUE_TEAM_SUP'
]]

red_team = clean_cells[[
    'RED_TEAM_TOP',
    'RED_TEAM_JNG',
    'RED_TEAM_MID',
    'RED_TEAM_ADC',
    'RED_TEAM_SUP'
]]

print(red_team)

In [ ]:
sln = []

for x in red_team.iterrows():
    team_list = [x[1][0], x[1][1], x[1][2], x[1][3], x[1][4]]
    result = []

    for name in team_list:
        result.append(str(name).replace("\'", "").lower())

    sln.append(result)

new_sln = pd.DataFrame(sln)

champion_names.reverse()

shurima_dict = []

for x in df.iterrows():
    shurima_dict.append((x[1][2], int(x[1][1])))

sundisk = dict(shurima_dict)

new_red_team = new_sln.replace(sundisk)

print(new_red_team)


with open("champion_dictionary.json", mode="w") as file:
    json.dump(sundisk, file, indent=4)